In [1]:
# EA Antibacterial GNN Dataset — East Africa focus (Tanzania prioritized)
#
# This notebook orchestrates data ingestion from multiple sources (EANPDB/ANPDB, NPASS, ChEMBL, PubChem),
# applies an East Africa filter (with Tanzania priority), canonicalizes compounds (RDKit),
# harmonizes antibacterial labels (MIC/IC50 → µM), and exports GNN-ready artifacts.
#
# It includes robust network/auth setup:
# - Basic auth, Bearer tokens, or .netrc
# - Corporate proxies (HTTP(S)_PROXY)
# - Retries and connectivity checks
#
# Outputs (in ./data/outputs):
# - gnn_dataset_flat.csv, nodes_compounds.csv, edges.csv, dataset_summary.json
# - Optional: PyTorch Geometric graph shards (graphs_shard_*.pt) if torch+pyg installed


In [2]:
# 1) Imports and environment configuration
import os
import json
import time
from pathlib import Path
from datetime import datetime

import pandas as pd

# Local helper module created for session
import ea_gnn_data as ea

DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "raw"
OUT_DIR = DATA_DIR / "outputs"
for d in (RAW_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("Data directories ready:", DATA_DIR.resolve())


Data directories ready: C:\Users\Thomas\Downloads\Thomas_Marco_230242445757\data


In [3]:
# 2) Authentication & network setup
# Configure via environment variables for flexibility. Examples:
# - AUTH_MODE=basic AUTH_USER=username AUTH_PASS=pass
# - AUTH_MODE=bearer AUTH_TOKEN=... (for endpoints that require it)
# - AUTH_MODE=netrc (use ~/.netrc)
# - HTTP(S)_PROXY / NO_PROXY for proxies

AUTH_MODE = os.getenv("AUTH_MODE", "none").lower()
AUTH_USER = os.getenv("AUTH_USER")
AUTH_PASS = os.getenv("AUTH_PASS")
AUTH_TOKEN = os.getenv("AUTH_TOKEN")
HTTP_PROXY = os.getenv("HTTP_PROXY") or os.getenv("http_proxy")
HTTPS_PROXY = os.getenv("HTTPS_PROXY") or os.getenv("https_proxy")

proxies = None
if HTTP_PROXY or HTTPS_PROXY:
    proxies = {}
    if HTTP_PROXY:
        proxies["http"] = HTTP_PROXY
    if HTTPS_PROXY:
        proxies["https"] = HTTPS_PROXY

cfg = ea.AuthConfig(
    mode=AUTH_MODE,
    username=AUTH_USER,
    password=AUTH_PASS,
    token=AUTH_TOKEN,
    proxies=proxies,
    timeout=int(os.getenv("HTTP_TIMEOUT", "45")),
    retries=int(os.getenv("HTTP_RETRIES", "5")),
    backoff=float(os.getenv("HTTP_BACKOFF", "0.5")),
    user_agent=os.getenv("HTTP_UA", "EA-GNN-Dataset/1.0 (+https://example.org)")
)
session = ea.build_session(cfg)
print("Session ready. Auth mode:", cfg.mode, "Proxies:", bool(cfg.proxies))

connectivity = ea.check_connectivity(session)
print("Connectivity check:")
print(json.dumps(connectivity, indent=2))

# If key hosts are blocked (401/403/407), provide actionable hints
blocked = {k: v for k, v in connectivity.items() if not v.get("ok")}
if blocked:
    print("\nSome endpoints not reachable:")
    for k, info in blocked.items():
        print(f" - {k}: {info}")
    print("\nTips: If behind a corporate proxy, set HTTP_PROXY/HTTPS_PROXY.\n"
          "If auth is required, try AUTH_MODE=basic or AUTH_MODE=bearer.\n"
          "For machine/user-account based auth, set AUTH_MODE=netrc and populate ~/.netrc.")


Session ready. Auth mode: none Proxies: False
Connectivity check:
{
  "ebi_latest": {
    "ok": true,
    "status": 200
  },
  "chembl_api": {
    "ok": true,
    "status": 200
  },
  "pubchem": {
    "ok": true,
    "status": 200
  },
  "npass": {
    "ok": true,
    "status": 200
  }
}


In [7]:
# 3) Configuration — East Africa scope and run limits
EA_COUNTRIES = [
    "Tanzania",  # priority
    "Kenya", "Uganda", "Ethiopia", "Rwanda", "Burundi",
    "Somalia", "South Sudan"
]
PRIORITY_COUNTRY = "Tanzania"

# Dry-run limits (increase for full run)
LIMIT_CHEMBL_ACTIVITIES = int(os.getenv("LIMIT_CHEMBL_ACTIVITIES", "200000"))  # per paging
LIMIT_NPASS = int(os.getenv("LIMIT_NPASS", "-1"))  # -1 means all

# Label thresholds
ACTIVE_MIC_UM = float(os.getenv("ACTIVE_MIC_UM", "10"))  # used for binarization

print("EA countries:", EA_COUNTRIES)
print("Priority:", PRIORITY_COUNTRY)


EA countries: ['Tanzania', 'Kenya', 'Uganda', 'Ethiopia', 'Rwanda', 'Burundi', 'Somalia', 'South Sudan']
Priority: Tanzania


In [5]:
# 4) Optional: Resolve and (if desired) download latest ChEMBL SDF/PG dump
try:
    url, fname = ea.chembl_resolve_latest("sdf", session=session)
    print("Latest ChEMBL SDF:", url)
    # NOTE: Large download — uncomment to fetch
    # r = session.get(url, stream=True)
    # r.raise_for_status()
    # with open(fname, "wb") as f:
    #     for chunk in r.iter_content(chunk_size=1024*1024):
    #         if chunk:
    #             f.write(chunk)
    # print("Saved:", fname)
except Exception as e:
    print("ChEMBL latest resolve failed:", e)


Latest ChEMBL SDF: https://ftp.ebi.ac.uk/pub/databases/chembl/ChEMBLdb/latest/chembl_37.sdf.gz


In [6]:
# 5) NPASS — Download raw tables (HTTP; often works without auth)
# Show exactly which URLs will be used and their reachability before downloading
try:
    import os as _os
    from itertools import islice as _islice

    env_urls = {
        "compound": _os.getenv("NPASS_URL_COMPOUND"),
        "structure": _os.getenv("NPASS_URL_STRUCTURE"),
        "activity": _os.getenv("NPASS_URL_ACTIVITY"),
        "species": _os.getenv("NPASS_URL_SPECIES"),
    }
    defaults = list(getattr(ea, "NPASS_FILES", []))
    selected = [u for u in [env_urls["compound"], env_urls["structure"], env_urls["activity"], env_urls["species"]] if u] or defaults

    print("NPASS selected URLs (env overrides first, else defaults):")
    for u in selected:
        print(" -", u)

    # Quick reachability check (HEAD -> GET fallback) for visibility
    print("\nConnectivity to NPASS targets:")
    for u in selected:
        try:
            h = session.head(u)
            code = h.status_code
            if code >= 400:
                g = session.get(u, stream=True)
                code = g.status_code
                if hasattr(g, "close"):
                    g.close()
            print(f" - {u} : HTTP {code}")
        except Exception as ce:
            print(f" - {u} : ERROR {ce}")

    # Perform the actual downloads
    out_dir = str(RAW_DIR / "npass")
    saved = ea.npass_download_all(out_dir, session=session)
    print("\nNPASS files saved:")
    for p in saved:
        try:
            size_b = (Path(p).stat().st_size if Path(p).exists() else 0)
            print(f" - {p} ({size_b} bytes)")
        except Exception:
            print(f" - {p}")

    # Preview the first few lines of each text/CSV file
    for path in saved:
        if any(path.lower().endswith(ext) for ext in (".txt", ".csv", ".tsv")):
            print("\nPreview:", path)
            try:
                with open(path, "r", encoding="utf-8", errors="ignore") as f:
                    head = list(_islice(f, 3))
                for ln in head:
                    print(ln.rstrip("\n"))
            except Exception as pe:
                print("(Could not preview)", pe)
except Exception as e:
    print("NPASS download failed:", e)
    print("Remedies:\n"
          "  1) Open https://bidd.group/NPASS/ and copy the current download links.\n"
          "  2) If links changed, add them to NPASS_DISCOVERY_PAGES in ea_gnn_data.py.\n"
          "  3) If the site blocks scripted downloads, fetch files manually to data/raw/npass/.\n"
          "  4) If a corporate proxy is required, set HTTP_PROXY/HTTPS_PROXY and re-run.")


NPASS selected URLs (env overrides first, else defaults):
 - https://bidd.group/NPASS/downloadFiles/NPASS3.0_naturalproducts_generalinfo.txt
 - https://bidd.group/NPASS/downloadFiles/NPASS3.0_naturalproducts_structure.txt
 - https://bidd.group/NPASS/downloadFiles/NPASS3.0_activities.txt
 - https://bidd.group/NPASS/downloadFiles/NPASS3.0_naturalproducts_species_pair.txt

Connectivity to NPASS targets:
 - https://bidd.group/NPASS/downloadFiles/NPASS3.0_naturalproducts_generalinfo.txt : HTTP 200
 - https://bidd.group/NPASS/downloadFiles/NPASS3.0_naturalproducts_structure.txt : HTTP 200
 - https://bidd.group/NPASS/downloadFiles/NPASS3.0_activities.txt : HTTP 200
 - https://bidd.group/NPASS/downloadFiles/NPASS3.0_naturalproducts_species_pair.txt : HTTP 200


KeyboardInterrupt: 

KeyboardInterrupt: 

In [8]:
# 6) ChEMBL antibacterial activities — stream a sample
from tqdm import tqdm

chembl_rows = []
try:
    for rec in tqdm(ea.chembl_iter_antibacterial_activities(session, limit=LIMIT_CHEMBL_ACTIVITIES)):
        chembl_rows.append(rec)
    print("ChEMBL antibacterial records fetched:", len(chembl_rows))
except PermissionError as pe:
    print("ChEMBL requires authentication or IP allow-listing:", pe)
except Exception as e:
    print("ChEMBL fetch error:", e)

chembl_df = pd.DataFrame(chembl_rows)
print("chembl_df columns:", list(chembl_df.columns)[:20])


94743it [4:58:44,  5.29it/s]


ChEMBL fetch error: HTTPSConnectionPool(host='www.ebi.ac.uk', port=443): Max retries exceeded with url: /chembl/api/data/activity.json?limit=1000&offset=2289000 (Caused by ReadTimeoutError("HTTPSConnectionPool(host='www.ebi.ac.uk', port=443): Read timed out. (read timeout=45)"))
chembl_df columns: ['action_type', 'activity_comment', 'activity_id', 'activity_properties', 'assay_chembl_id', 'assay_description', 'assay_type', 'assay_variant_accession', 'assay_variant_mutation', 'bao_endpoint', 'bao_format', 'bao_label', 'canonical_smiles', 'data_validity_comment', 'data_validity_description', 'document_chembl_id', 'document_journal', 'document_year', 'ligand_efficiency', 'modality']


In [12]:
# import requests, json
# CHEMBL_BASE = "https://www.ebi.ac.uk/chembl/api/data"
# url = f"{CHEMBL_BASE}/activity.json"
# r = requests.get(url, params={"limit": 1}, timeout=30)
# r.raise_for_status()
# js = r.json()
# print("status:", r.status_code)
# print("sample next:", js.get("page_meta", {}).get("next"))
# print(json.dumps(js.get("activities", [])[:1], indent=2)[:800])


# 1) Show all columns (sorted)
sorted_cols = sorted(chembl_df.columns)
print(len(sorted_cols))
print(sorted_cols)

# 2) Check common alternatives for organism and relation
cands_org = [c for c in chembl_df.columns if 'organism' in c.lower()]
print('organism-like columns:', cands_org)

cands_rel = [c for c in chembl_df.columns if 'relation' in c.lower()]
print('relation-like columns:', cands_rel)

# 3) Check which standard_* fields exist
cands_std = [c for c in chembl_df.columns if c.lower().startswith('standard_')]
print('standard_* columns:', cands_std[:20])

47
['action_type', 'activity_comment', 'activity_id', 'activity_properties', 'assay_chembl_id', 'assay_description', 'assay_type', 'assay_variant_accession', 'assay_variant_mutation', 'bao_endpoint', 'bao_format', 'bao_label', 'canonical_smiles', 'data_validity_comment', 'data_validity_description', 'document_chembl_id', 'document_journal', 'document_year', 'ligand_efficiency', 'modality', 'molecule_chembl_id', 'molecule_pref_name', 'parent_molecule_chembl_id', 'pchembl_value', 'potential_duplicate', 'qudt_units', 'record_id', 'relation', 'src_id', 'standard_flag', 'standard_relation', 'standard_text_value', 'standard_type', 'standard_units', 'standard_upper_value', 'standard_value', 'target_chembl_id', 'target_organism', 'target_pref_name', 'target_tax_id', 'text_value', 'toid', 'type', 'units', 'uo_units', 'upper_value', 'value']
organism-like columns: ['target_organism']
relation-like columns: ['relation', 'standard_relation']
standard_* columns: ['standard_flag', 'standard_relation

In [14]:
import pandas as pd

# Preferred→fallback column choices
ORG_COLS = ['assay_organism', 'target_organism']
REL_COLS = ['relation', 'standard_relation']

# helper to pick the first column that exists
first_present = lambda cols: next((c for c in cols if c in chembl_df.columns), None)

col_org = first_present(ORG_COLS)
col_rel = first_present(REL_COLS)

need_base = ['standard_type','standard_value','standard_units']
view_cols = ['molecule_chembl_id','canonical_smiles'] + [c for c in need_base if c in chembl_df.columns]
if col_rel: view_cols.append(col_rel)
if col_org: view_cols.append(col_org)

chembl_view = chembl_df[view_cols].copy()
if col_org: chembl_view.rename(columns={col_org: 'organism'}, inplace=True)
if col_rel: chembl_view.rename(columns={col_rel: 'relation'}, inplace=True)

# Normalize standard_type for filtering
if 'standard_type' in chembl_view.columns:
    chembl_view['standard_type_u'] = chembl_view['standard_type'].astype(str).str.upper()
else:
    chembl_view['standard_type_u'] = ''

print('chembl_view columns:', list(chembl_view.columns))

TYPES = ["MIC","IC50","EC50","POTENCY"]  # compare uppercase
ORG_PATTERNS = [
    'Escherichia coli','Staphylococcus aureus','Klebsiella pneumoniae',
    'Acinetobacter baumannii','Pseudomonas aeruginosa',
    'Mycobacterium tuberculosis','Salmonella','Shigella',
    'Vibrio cholerae','Streptococcus pneumoniae','Enterococcus faecium'
]

org_series = chembl_view['organism'].astype(str) if 'organism' in chembl_view.columns else pd.Series('', index=chembl_view.index)
mask_type = chembl_view['standard_type_u'].isin(TYPES)
mask_org = org_series.str.contains('|'.join(ORG_PATTERNS), case=False, na=False)

chembl_anti = chembl_view[mask_type & mask_org].copy()
print('Antibacterial rows:', len(chembl_anti))
chembl_anti.head(3)



# Try to compute molecular weight from SMILES if possible
if 'canonical_smiles' in chembl_anti.columns:
    try:
        from rdkit import Chem
        from rdkit.Chem import Descriptors
        def mw_from_smiles(s):
            if not isinstance(s, str) or not s: return None
            m = Chem.MolFromSmiles(s)
            return Descriptors.MolWt(m) if m else None
        chembl_anti['mw'] = chembl_anti['canonical_smiles'].map(mw_from_smiles)
    except Exception:
        chembl_anti['mw'] = None
else:
    chembl_anti['mw'] = None

# Conversion function (nM→µM, µM→µM, mg/L or ug/mL require MW)
def to_uM(row):
    v = row.get('standard_value')
    u = str(row.get('standard_units') or '').lower()
    mw = row.get('mw')
    if pd.isna(v):
        return None
    if u in ('nm','nanomolar'):
        return float(v)/1000.0
    if u in ('um','µm','micromolar','micro molar'):
        return float(v)
    if u in ('mg/l','ug/ml','µg/ml','mcg/ml'):
        if pd.notna(mw) and mw > 0:
            mg_per_l = float(v)  # ug/mL == mg/L numerically
            return (mg_per_l / (mw/1000.0))
        return None
    return None

chembl_anti['MIC_uM'] = chembl_anti.apply(to_uM, axis=1)
chembl_anti[['standard_type','standard_units','relation','organism','MIC_uM']].head(5)

# Choose an identity key; prefer InChIKey if you plan to compute it.
ID_COL = 'molecule_chembl_id'  # later we can switch to InChIKey

# Keep only rows with a computed MIC_uM (others can be handled separately)
has_mic = chembl_anti.dropna(subset=['MIC_uM'])

# Min (best) MIC per compound×organism
agg = (has_mic.groupby([ID_COL, 'organism'])['MIC_uM']
              .agg(['count','min','median'])
              .reset_index()
              .rename(columns={'min':'MIC_uM_min','median':'MIC_uM_median','count':'n_obs'}))

# Binary activity by threshold (default 10 µM)
ACTIVE_MIC_UM = 10.0
agg['active_le_10uM'] = agg['MIC_uM_min'] <= ACTIVE_MIC_UM
agg.head(10)

try:
    from rdkit import Chem
    def inchikey_from_smiles(s):
        if not isinstance(s, str) or not s: return None
        m = Chem.MolFromSmiles(s)
        return Chem.MolToInchiKey(m) if m else None
    # compute once per molecule_chembl_id
    smi_by_id = chembl_anti.dropna(subset=['canonical_smiles']).drop_duplicates([ID_COL])[[ID_COL,'canonical_smiles']]
    smi_by_id['inchikey'] = smi_by_id['canonical_smiles'].map(inchikey_from_smiles)
    agg = agg.merge(smi_by_id[[ID_COL,'inchikey']], on=ID_COL, how='left')
except Exception:
    pass

chembl_view columns: ['molecule_chembl_id', 'canonical_smiles', 'standard_type', 'standard_value', 'standard_units', 'relation', 'organism', 'standard_type_u']
Antibacterial rows: 94743


In [ ]:
# 7-Prep) Auto-synthesize literature_ea_curated.csv from NPASS 3.0 (EA-only; plants only)
# Builds a lightweight literature CSV at project root using NPASS species pairs + structures.
# - Filters to East Africa countries and plant kingdom (Plantae)
# - Excludes entries without country (per user default)
# - Adds provenance fields for traceability
try:
    import pandas as _pd
    from pathlib import Path as _Path
    from datetime import datetime as _dt

    _ROOT = _Path('.')
    _RAW_NPASS = _Path('data/raw/npass')
    _OUT_CSV = _ROOT / 'literature_ea_curated.csv'

    _EA_COUNTRIES = [
        'Tanzania','Kenya','Uganda','Ethiopia','Rwanda','Burundi','Somalia','South Sudan'
    ]

    def _read_tsv(_p: _Path) -> _pd.DataFrame:
        df = _pd.read_csv(_p, sep='\t', low_memory=False)
        df.columns = [c.strip() for c in df.columns]
        return df

    _FN_GENERAL = _RAW_NPASS / 'NPASS3.0_naturalproducts_generalinfo.txt'
    _FN_STRUCT  = _RAW_NPASS / 'NPASS3.0_naturalproducts_structure.txt'
    _FN_SPECIES = _RAW_NPASS / 'NPASS3.0_naturalproducts_species_pair.txt'
    _FN_SINFO   = _RAW_NPASS / 'NPASS3.0_species_info.txt'

    if _FN_GENERAL.exists() and _FN_STRUCT.exists() and _FN_SPECIES.exists():
        _ng = _read_tsv(_FN_GENERAL)
        _ns = _read_tsv(_FN_STRUCT)
        _npairs = _read_tsv(_FN_SPECIES)
        _nsinfo = _read_tsv(_FN_SINFO) if _FN_SINFO.exists() else _pd.DataFrame()

        _col_np_id_g = next((c for c in _ng.columns if c.lower() in ('np_id','npid','np id')), 'np_id')
        _col_name    = next((c for c in _ng.columns if c.lower() in ('pref_name','name','compound_name','preferred_name')), 'pref_name')

        _col_np_id_s = next((c for c in _ns.columns if c.lower() in ('np_id','npid','np id')), 'np_id')
        _col_smiles  = next((c for c in _ns.columns if c.strip().lower() in ('smiles','canonical_smiles')), 'SMILES' if 'SMILES' in _ns.columns else None)
        _col_ikey    = next((c for c in _ns.columns if c.strip().lower() in ('inchikey','inchi_key','inchikey_key')), 'InChIKey' if 'InChIKey' in _ns.columns else None)

        _col_np_id_p   = next((c for c in _npairs.columns if c.lower() in ('np_id','npid','np id')), 'np_id')
        _col_species_p = next((c for c in _npairs.columns if any(k in c.lower() for k in ['species','organism','plant'])), None)
        _col_ref_p     = next((c for c in _npairs.columns if any(k in c.lower() for k in ['ref','reference'])), None)

        if not _nsinfo.empty:
            _col_species_i = next((c for c in _nsinfo.columns if any(k in c.lower() for k in ['species','organism','scientific'])), None)
            _col_kingdom   = next((c for c in _nsinfo.columns if 'kingdom' in c.lower()), None)
            _col_country   = next((c for c in _nsinfo.columns if any(k in c.lower() for k in ['country','countries','distribution','geograph'])), None)
        else:
            _col_species_i = _col_kingdom = _col_country = None

        _g = _ng[[_col_np_id_g, _col_name]].dropna().rename(columns={_col_np_id_g:'np_id', _col_name:'compound_name'})
        _keep_s = ['np_id'] + ([_col_smiles] if _col_smiles else []) + ([_col_ikey] if _col_ikey else [])
        _s = _ns[[_col_np_id_s] + ([
            _col_smiles] if _col_smiles else []) + ([
            _col_ikey] if _col_ikey else [])].dropna(subset=[_col_np_id_s]).rename(columns={_col_np_id_s:'np_id', (_col_smiles or 'SMILES'):'smiles', (_col_ikey or 'InChIKey'):'inchikey'})
        _p = _npairs[[_col_np_id_p] + ([
            _col_species_p] if _col_species_p else []) + ([
            _col_ref_p] if _col_ref_p else [])].dropna(subset=[_col_np_id_p]).rename(columns={_col_np_id_p:'np_id', (_col_species_p or 'species'):'species', (_col_ref_p or 'reference'):'reference'})

        if _col_species_i and (_col_kingdom or _col_country):
            _keep_cols = [_col_species_i] + [c for c in [_col_kingdom, _col_country] if c]
            _si = _nsinfo[_keep_cols].drop_duplicates().rename(columns={_col_species_i:'species', (_col_kingdom or 'kingdom'):'kingdom', (_col_country or 'country'):'country'})
        else:
            # Fallback: species_info not available; proceed with unknown placeholders
            _si = _pd.DataFrame(columns=['species','kingdom','country'])

        # Join pairs with species info if any; otherwise carry unknown placeholders
        if not _si.empty:
            _pc = _p.merge(_si, on='species', how='left')
        else:
            _pc = _p.copy()
            _pc['kingdom'] = 'unknown'
            _pc['country'] = 'unknown'

        # Plants-only filter if kingdom is available; otherwise skip (keep unknown)
        if 'kingdom' in _pc.columns and _pc['kingdom'].notna().any():
            _mask_plants = _pc['kingdom'].astype(str).str.contains('plantae', case=False, na=False)
            _pc = _pc[_mask_plants]

        # East Africa country filter if country is available; otherwise defer (will remain 'unknown')
        if 'country' in _pc.columns and _pc['country'].notna().any():
            _pc['country_norm'] = _pc['country'].astype(str)
            _pc = _pc.assign(country_split=_pc['country_norm'].str.split(r'[;,\/]')).explode('country_split')
            _pc['country_split'] = _pc['country_split'].str.strip()
            _pc = _pc[_pc['country_split'].isin(_EA_COUNTRIES)]
            _pc = _pc.rename(columns={'country_split':'country'})
            _pc = _pc.drop(columns=['country_norm'])

        _pcs = _pc.merge(_g, on='np_id', how='inner').merge(_s, on='np_id', how='left')

        _out_cols = ['plant_name','country','compound_name','smiles','reference','notes','source_doi','source_url','year','curator']
        _base_url = 'https://bidd.group/NPASS/downloadFiles/'
        _rows = []
        for _, _r in _pcs.iterrows():
            _rows.append({
                'plant_name': _r.get('species'),
                'country': _r.get('country'),
                'compound_name': _r.get('compound_name'),
                'smiles': _r.get('smiles'),
                'reference': _r.get('reference') if 'reference' in _pcs.columns else 'NPASS 3.0 species pair',
                'notes': 'EA (NPASS 3.0 inferred); antibacterial potential unknown',
                'source_doi': '',
                'source_url': _base_url,
                'year': '',
                'curator': f'auto_synthesized@{_dt.utcnow().date().isoformat()}',
            })
        _lit = _pd.DataFrame(_rows, columns=_out_cols)
        _before = len(_lit)
        _lit = _lit.drop_duplicates(subset=['plant_name','country','compound_name']).reset_index(drop=True)
        print(f'Built literature_ea_curated.csv: {len(_lit)} rows (dedup from {_before})')
        print(' - Tanzania rows:', int((_lit['country'] == 'Tanzania').sum()))
        try:
            print(' - Countries present:', sorted(_lit['country'].dropna().unique().tolist()))
        except Exception:
            pass
        print('\nPreview:')
        print(_lit.head(10))
        _lit.to_csv(_OUT_CSV, index=False)
        print('\nSaved to:', _OUT_CSV.resolve())
    else:
        print('NPASS files not found under data/raw/npass/. Skipping auto-synthesis step.')
except Exception as __e:
    print('Auto-synthesis step failed:', __e)


In [28]:

from pathlib import Path
print("cwd:", Path.cwd())
p = Path("literature_ea_curated.csv")
print("LIT_PATH:", p.resolve(), "exists:", p.exists())

from pathlib import Path
print("cwd:", Path.cwd())
p = Path("literature_ea_curated.csv")
print("LIT_PATH:", p.resolve(), "exists:", p.exists())

from pathlib import Path
npass_dir = Path("data/raw/npass")
for name in [
    "NPASS3.0_naturalproducts_generalinfo.txt",
    "NPASS3.0_naturalproducts_structure.txt",
    "NPASS3.0_naturalproducts_species_pair.txt",
    "NPASS3.0_species_info.txt",
]:
    p = npass_dir / name
    print(name, "→", p.resolve(), "exists:", p.exists())
    
    print(EA_COUNTRIES, "priority:", PRIORITY_COUNTRY)

cwd: C:\Users\Thomas\Downloads\Thomas_Marco_230242445757
LIT_PATH: C:\Users\Thomas\Downloads\Thomas_Marco_230242445757\literature_ea_curated.csv exists: False
cwd: C:\Users\Thomas\Downloads\Thomas_Marco_230242445757
LIT_PATH: C:\Users\Thomas\Downloads\Thomas_Marco_230242445757\literature_ea_curated.csv exists: False
NPASS3.0_naturalproducts_generalinfo.txt → C:\Users\Thomas\Downloads\Thomas_Marco_230242445757\data\raw\npass\NPASS3.0_naturalproducts_generalinfo.txt exists: True
['Tanzania', 'Kenya', 'Uganda', 'Ethiopia', 'Rwanda', 'Burundi', 'Somalia', 'South Sudan'] priority: Tanzania
NPASS3.0_naturalproducts_structure.txt → C:\Users\Thomas\Downloads\Thomas_Marco_230242445757\data\raw\npass\NPASS3.0_naturalproducts_structure.txt exists: True
['Tanzania', 'Kenya', 'Uganda', 'Ethiopia', 'Rwanda', 'Burundi', 'Somalia', 'South Sudan'] priority: Tanzania
NPASS3.0_naturalproducts_species_pair.txt → C:\Users\Thomas\Downloads\Thomas_Marco_230242445757\data\raw\npass\NPASS3.0_naturalproducts_sp

In [23]:
# 7) Literature/EANPDB/ANPDB stubs
# In restricted environments, you may preload a literature CSV with columns:
# plant_name, country, compound_name, smiles(optional), reference, notes
# If smiles missing, we attempt PubChem resolution below.

LIT_PATH = Path("literature_ea_curated.csv")
if LIT_PATH.exists():
    lit_df = pd.read_csv(LIT_PATH)
    print("Loaded literature records:", len(lit_df))
else:
    lit_df = pd.DataFrame(columns=["plant_name","country","compound_name","smiles","reference","notes"])
    print("No literature_ea_curated.csv found; proceeding with empty frame.")


No literature_ea_curated.csv found; proceeding with empty frame.


In [17]:
# 8) PubChem name → SMILES enrichment for literature rows without SMILES
if not lit_df.empty:
    if "smiles" not in lit_df.columns:
        lit_df["smiles"] = None
    mask = lit_df["smiles"].isna() | (lit_df["smiles"].astype(str).str.len() == 0)
    unresolved = lit_df.loc[mask, "compound_name"].dropna().unique().tolist()
    print("Need to resolve SMILES for", len(unresolved), "names")
    resolved = {}
    for name in tqdm(unresolved):
        try:
            smi = ea.pubchem_name_to_smiles(name, session=session)
            if smi:
                resolved[name] = smi
        except Exception:
            pass
    if resolved:
        lit_df.loc[mask, "smiles"] = lit_df.loc[mask, "compound_name"].map(resolved)
        print("Resolved via PubChem:", len(resolved))


In [18]:
# 9) Merge sources to a compound table with canonicalization and InChIKey
try:
    from rdkit import Chem  # local import for clearer error message
except Exception as e:
    print("RDKit not installed. Install with e.g.:\n  pip install rdkit-pypi\nThen re-run this cell.")
    Chem = None

records = []
# From literature (EA-filtered by country)
if not lit_df.empty:
    ea_lit = lit_df[lit_df["country"].isin(EA_COUNTRIES)].copy()
    for _, row in ea_lit.iterrows():
        smi = row.get("smiles")
        if not smi or smi != smi:
            continue
        try:
            can = ea.canonicalize_smiles(smi)
            ik = ea.inchikey_from_smiles(can) if can else None
            rec = {
                "compound_name": row.get("compound_name"),
                "smiles": can or smi,
                "inchikey": ik,
                "source": "Literature",
                "source_url": row.get("reference"),
                "retrieved_at": datetime.utcnow().isoformat(),
                "country": row.get("country"),
                "tanzania_flag": str(row.get("country") == PRIORITY_COUNTRY),
            }
            records.append(rec)
        except Exception:
            continue

compounds_df = pd.DataFrame(records)
print("Initial EA compounds:", len(compounds_df))

# Deduplicate by InChIKey (if available), else by canonical SMILES
if not compounds_df.empty:
    if "inchikey" in compounds_df.columns:
        before = len(compounds_df)
        compounds_df = compounds_df.sort_values("inchikey").drop_duplicates(subset=["inchikey","smiles"], keep="first")
        print(f"Dedup: {before} -> {len(compounds_df)}")


Initial EA compounds: 0


In [19]:
# 10) Compute descriptors and build nodes/edges placeholders
if not compounds_df.empty:
    desc_rows = []
    for _, r in compounds_df.iterrows():
        smi = r["smiles"]
        try:
            desc = ea.descriptors_from_smiles(smi)
        except Exception as e:
            desc = {}
        row = {**r.to_dict(), **desc}
        desc_rows.append(row)
    nodes_compounds = pd.DataFrame(desc_rows)
else:
    nodes_compounds = pd.DataFrame()

# Simple edges from literature (plant→compound)
edges = pd.DataFrame(columns=["edge_type","plant_name","inchikey","evidence","source_url"]) 
if not lit_df.empty and not compounds_df.empty:
    merged = pd.merge(lit_df, compounds_df, left_on=["compound_name"], right_on=["compound_name"], how="inner")
    for _, rr in merged.iterrows():
        edges.loc[len(edges)] = [
            "plant_compound",
            rr.get("plant_name"),
            rr.get("inchikey"),
            rr.get("notes"),
            rr.get("reference"),
        ]


In [20]:
# 11) Build a simple flat training table (binary labels if no MIC yet)
# For now, create a placeholder active flag if present in literature notes (e.g., 'antibacterial')
flat = nodes_compounds.copy()
if not flat.empty:
    flat["label_active"] = flat.get("notes", "").astype(str).str.contains("antibacterial|antimicrobial", case=False, na=False)

# TODO: Integrate ChEMBL/NPASS quantitative MIC_uM once fetched and mapped by InChIKey


In [21]:
# 12) Save outputs and summary
outputs = {
    "gnn_dataset_flat.csv": flat,
    "nodes_compounds.csv": nodes_compounds,
    "edges.csv": edges,
}

for name, df in outputs.items():
    path = OUT_DIR / name
    if isinstance(df, pd.DataFrame):
        df.to_csv(path, index=False)
        print("Saved:", path)

summary = {
    "generated_at": datetime.utcnow().isoformat(),
    "east_africa_countries": EA_COUNTRIES,
    "priority_country": PRIORITY_COUNTRY,
    "total_records_flat": int(len(flat)) if isinstance(flat, pd.DataFrame) else 0,
    "unique_compounds": int(flat["inchikey"].nunique()) if (isinstance(flat, pd.DataFrame) and "inchikey" in flat.columns) else 0,
    "sources": ["Literature", "NPASS", "ChEMBL", "PubChem"],
    "connectivity": connectivity,
}
with open(OUT_DIR / "dataset_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
print("Saved:", OUT_DIR / "dataset_summary.json")


Saved: data\outputs\gnn_dataset_flat.csv
Saved: data\outputs\nodes_compounds.csv
Saved: data\outputs\edges.csv
Saved: data\outputs\dataset_summary.json


C:\Users\Thomas\AppData\Local\Temp\ipykernel_18688\796076419.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "generated_at": datetime.utcnow().isoformat(),


In [ ]:
# 13) Next steps (manual in notebook):
# - Increase LIMIT_CHEMBL_ACTIVITIES and re-run section 6 on an unrestricted network.
# - Load EANPDB/ANPDB tables (place a CSV in data/raw/anpdb and parse to plant/compound/SMILES) then re-run sections 9–12.
# - Map ChEMBL/NPASS quantitative MIC to InChIKey, convert to µM via ea.mic_to_uM (requires MW), and add regression labels to flat table.
# - Optionally, build PyG graphs from SMILES; save shards to data/outputs.


In [22]:
# 12b) Build ChEMBL labels → flat table, nodes, and edges; then save outputs
try:
    from datetime import datetime as _dt
    import json as _json
    import math as _math

    # Ensure we have an InChIKey for robust dedup/joins (if not already computed above)
    if 'inchikey' not in agg.columns:
        try:
            from rdkit import Chem as _Chem
            def _inchikey_from_smiles(_s):
                if not isinstance(_s, str) or not _s:
                    return None
                m = _Chem.MolFromSmiles(_s)
                return _Chem.MolToInchiKey(m) if m else None
            _smi_by_id = chembl_anti.dropna(subset=['canonical_smiles']).drop_duplicates([ID_COL])[[ID_COL,'canonical_smiles']]
            _smi_by_id['inchikey'] = _smi_by_id['canonical_smiles'].map(_inchikey_from_smiles)
            agg = agg.merge(_smi_by_id[[ID_COL,'inchikey']], on=ID_COL, how='left')
        except Exception:
            pass

    # Pathogen short-name mapping for column-friendly names
    _short = {
        'Escherichia coli': 'ecoli',
        'Staphylococcus aureus': 'saureus',
        'Klebsiella pneumoniae': 'kpneumoniae',
        'Acinetobacter baumannii': 'abaumannii',
        'Pseudomonas aeruginosa': 'paeruginosa',
        'Mycobacterium tuberculosis': 'mtb',
        'Salmonella': 'salmonella',
        'Shigella': 'shigella',
        'Vibrio cholerae': 'vcholerae',
        'Streptococcus pneumoniae': 'spneumoniae',
        'Enterococcus faecium': 'efaecium',
    }

    # Pivot MIC minima per organism
    _mic_pivot = agg.pivot_table(index=[ID_COL, 'inchikey'], columns='organism', values='MIC_uM_min', aggfunc='min')
    # Pivot activity flags per organism (any active across replicates)
    _act_pivot = agg.pivot_table(index=[ID_COL, 'inchikey'], columns='organism', values='active_le_10uM', aggfunc='max')

    # Flatten columns with short names
    def _flatten_cols(df, prefix):
        cols = {}
        for c in df.columns:
            key = _short.get(str(c), str(c).lower().replace(' ', '_'))
            cols[c] = f"{prefix}{key}"
        return df.rename(columns=cols)

    _mic_pivot = _flatten_cols(_mic_pivot, 'MIC_uM_min__')
    _act_pivot = _flatten_cols(_act_pivot, 'active_le_10uM__')

    flat = _mic_pivot.join(_act_pivot, how='outer').reset_index()

    # Attach a simple count of organisms with labels per compound
    _label_cols = [c for c in flat.columns if c.startswith('MIC_uM_min__')]
    flat['n_pathogens_labeled'] = flat[_label_cols].notna().sum(axis=1)

    # Bring over canonical_smiles (first available per ID) for convenience
    _id_smi = chembl_anti.dropna(subset=['canonical_smiles']).drop_duplicates([ID_COL])[[ID_COL,'canonical_smiles']]
    flat = flat.merge(_id_smi, on=ID_COL, how='left')

    # nodes_compounds: unique compounds with optional RDKit descriptors
    nodes_cols = ['inchikey', ID_COL, 'canonical_smiles']
    nodes_compounds = flat[nodes_cols].drop_duplicates().copy()

    # Try to compute descriptors (skip gracefully if RDKit missing)
    try:
        from rdkit import Chem as _Chem
        from rdkit.Chem import Descriptors as _Desc
        def _descriptors(_s):
            if not isinstance(_s, str) or not _s:
                return {}
            m = _Chem.MolFromSmiles(_s)
            if not m:
                return {}
            return {
                'MW': _Desc.MolWt(m),
                'XLogP': _Desc.MolLogP(m),
                'HBD': _Desc.NumHDonors(m),
                'HBA': _Desc.NumHAcceptors(m),
                'TPSA': _Desc.TPSA(m),
                'AtomCount': m.GetNumAtoms(),
                'RingCount': _Chem.GetSSSR(m),
            }
        _desc_rows = []
        for _, rr in nodes_compounds.iterrows():
            d = _descriptors(rr.get('canonical_smiles'))
            _desc_rows.append({**rr.to_dict(), **d})
        nodes_compounds = pd.DataFrame(_desc_rows)
    except Exception:
        pass

    # edges: compound → bacterium with MIC summary
    edges_rows = []
    for _, r in agg.iterrows():
        edges_rows.append({
            'edge_type': 'compound_bacterium',
            'molecule_chembl_id': r.get(ID_COL),
            'inchikey': r.get('inchikey'),
            'bacterium': r.get('organism'),
            'MIC_uM_min': r.get('MIC_uM_min'),
            'MIC_uM_median': r.get('MIC_uM_median'),
            'n_obs': r.get('n_obs'),
            'active_le_10uM': bool(r.get('active_le_10uM')) if r.get('active_le_10uM') is not None else None,
            'source': 'ChEMBL',
            'source_url': 'https://www.ebi.ac.uk/chembl/'
        })
    edges = pd.DataFrame(edges_rows)

    # Optional: quick NPASS linkage flag by InChIKey (if NPASS structure is present and RDKit computed inchikeys exist)
    npass_dir = RAW_DIR / 'npass'
    npass_struct = None
    for cand in ['NPASS3.0_naturalproducts_structure.txt','npass_structure.txt','structure.txt']:
        p = npass_dir / cand
        if p.exists():
            npass_struct = p
            break
    flat['npass_linked'] = False
    if npass_struct is not None:
        try:
            _ns = pd.read_csv(npass_struct, sep='\t', low_memory=False)
            # Use InChIKey column if present
            ik_col = None
            for c in _ns.columns:
                if c.strip().lower() in ('inchikey','inchi_key','inchikey_key'):
                    ik_col = c
                    break
            if ik_col:
                _ik_set = set(_ns[ik_col].dropna().astype(str).unique().tolist())
                if 'inchikey' in flat.columns:
                    flat['npass_linked'] = flat['inchikey'].astype(str).isin(_ik_set)
        except Exception:
            pass

    # Save outputs
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    flat_path = OUT_DIR / 'gnn_dataset_flat.csv'
    nodes_path = OUT_DIR / 'nodes_compounds.csv'
    edges_path = OUT_DIR / 'edges.csv'

    flat.to_csv(flat_path, index=False)
    nodes_compounds.to_csv(nodes_path, index=False)
    edges.to_csv(edges_path, index=False)

    # Summary JSON for slides
    summary = {
        'generated_at': _dt.utcnow().isoformat(),
        'total_chembl_rows_streamed': int(len(chembl_df)) if isinstance(chembl_df, pd.DataFrame) else None,
        'antibacterial_rows': int(len(chembl_anti)) if isinstance(chembl_anti, pd.DataFrame) else None,
        'rows_with_MIC_uM': int(len(has_mic)) if isinstance(has_mic, pd.DataFrame) else None,
        'unique_compounds_flat': int(flat['inchikey'].nunique()) if 'inchikey' in flat.columns else int(flat[ID_COL].nunique()),
        'n_nodes_compounds': int(len(nodes_compounds)),
        'n_edges': int(len(edges)),
        'n_pathogens_columns': len(_label_cols),
        'npass_linked_compounds': int(flat['npass_linked'].sum()) if 'npass_linked' in flat.columns else 0,
        'outputs': {
            'gnn_dataset_flat.csv': str(flat_path.resolve()),
            'nodes_compounds.csv': str(nodes_path.resolve()),
            'edges.csv': str(edges_path.resolve()),
        },
    }
    with open(OUT_DIR / 'dataset_summary.json', 'w', encoding='utf-8') as _f:
        _json.dump(summary, _f, indent=2)

    print('Saved outputs:')
    for k, v in summary['outputs'].items():
        try:
            import os as _os
            _sz = _os.path.getsize(v)
            print(f" - {k}: {v} ({_sz} bytes)")
        except Exception:
            print(f" - {k}: {v}")
    print('Summary saved to:', OUT_DIR / 'dataset_summary.json')

except Exception as _e:
    print('Export step failed:', _e)


Saved outputs:
 - gnn_dataset_flat.csv: C:\Users\Thomas\Downloads\Thomas_Marco_230242445757\data\outputs\gnn_dataset_flat.csv (985018 bytes)
 - nodes_compounds.csv: C:\Users\Thomas\Downloads\Thomas_Marco_230242445757\data\outputs\nodes_compounds.csv (1484901 bytes)
 - edges.csv: C:\Users\Thomas\Downloads\Thomas_Marco_230242445757\data\outputs\edges.csv (1058514 bytes)
Summary saved to: data\outputs\dataset_summary.json


C:\Users\Thomas\AppData\Local\Temp\ipykernel_18688\850922863.py:148: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'generated_at': _dt.utcnow().isoformat(),
